# DA6401 Assignment 2 — W&B Report Generation

## What this notebook does
- Runs ablation experiments (Sections 2.1, 2.2)
- Generates all visualizations (Sections 2.4, 2.5, 2.6, 2.7)

⚠️ Before running:
- Enable GPU: Settings → Accelerator → GPU P100
- Enable Internet: Settings → Internet → On
- Add datasets: pet_dataset + classifier-checkpoint

## CELL 1 — Setup

In [ ]:
import os, sys, subprocess, torch

# Install packages
subprocess.run(['pip', 'install', '-q', 'albumentations', 'wandb', 'scikit-learn'], check=True)

# Paths
WORK_DIR    = '/kaggle/working'
PROJECT_DIR = f'{WORK_DIR}/DL6402_Assignment2'
CKPT_DIR    = f'{WORK_DIR}/checkpoints'
LOCAL_DATA  = f'{WORK_DIR}/pets'
INET_IMGS   = f'{WORK_DIR}/internet_images'

for d in [CKPT_DIR, LOCAL_DATA, INET_IMGS]:
    os.makedirs(d, exist_ok=True)

# Clone repo
if not os.path.exists(PROJECT_DIR):
    os.system(f'git clone https://github.com/Ganesh2024/DL6402_Assignment2.git {PROJECT_DIR}')
else:
    os.system(f'cd {PROJECT_DIR} && git pull')

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# GPU check
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None!')
print('Setup done!')

## CELL 2 — Extract dataset

In [ ]:
import os, time

LOCAL_DATA = '/kaggle/working/pets'

if os.path.exists(os.path.join(LOCAL_DATA, 'images')):
    n = len(os.listdir(os.path.join(LOCAL_DATA, 'images')))
    print(f'Already extracted ({n} images)')
else:
    # Find dataset in kaggle input
    for item in os.listdir('/kaggle/input'):
        candidate = f'/kaggle/input/{item}/dataset'
        if os.path.exists(candidate):
            print(f'Copying from {candidate}...')
            t0 = time.time()
            os.system(f'cp -r {candidate}/* {LOCAL_DATA}/')
            print(f'Done in {time.time()-t0:.0f}s')
            break
        # Check for zip
        for sub in os.listdir(f'/kaggle/input/{item}'):
            if sub.endswith('.zip'):
                zpath = f'/kaggle/input/{item}/{sub}'
                print(f'Extracting {zpath}...')
                t0 = time.time()
                os.system(f'unzip -q "{zpath}" -d /kaggle/working/tmp_ext')
                inner = '/kaggle/working/tmp_ext/dataset'
                src = inner if os.path.exists(inner) else '/kaggle/working/tmp_ext'
                os.system(f'mv {src}/* {LOCAL_DATA}/')
                os.system('rm -rf /kaggle/working/tmp_ext')
                print(f'Done in {time.time()-t0:.0f}s')
                break

    n = len(os.listdir(os.path.join(LOCAL_DATA, 'images')))
    print(f'Images: {n}')
print('Dataset ready ✅')

## CELL 3 — Copy classifier.pth checkpoint

In [ ]:
import os, torch

CKPT_DIR = '/kaggle/working/checkpoints'

# Search for classifier.pth in kaggle input
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'classifier.pth':
            src = os.path.join(root, f)
            dst = os.path.join(CKPT_DIR, 'classifier.pth')
            os.system(f'cp "{src}" "{dst}"')
            print(f'Copied from {src}')
            break

# Also copy localizer and unet if available
for fname in ['localizer.pth', 'unet.pth', 'unet_frozen.pth', 'unet_partial.pth']:
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f == fname:
                src = os.path.join(root, f)
                dst = os.path.join(CKPT_DIR, fname)
                os.system(f'cp "{src}" "{dst}"')
                print(f'Copied {fname}')

# Status
print('\nCheckpoint status:')
for fname in ['classifier.pth', 'localizer.pth', 'unet.pth']:
    fp = os.path.join(CKPT_DIR, fname)
    if os.path.exists(fp):
        c = torch.load(fp, map_location='cpu')
        print(f'  ✅ {fname} epoch={c.get("epoch")} metric={c.get("best_metric",0):.4f}')
    else:
        print(f'  ❌ {fname} not found')

## CELL 4 — W&B Login

In [ ]:
import os, wandb

WANDB_KEY = 'YOUR_WANDB_API_KEY_HERE'
os.environ['WANDB_API_KEY'] = WANDB_KEY
wandb.login(key=WANDB_KEY, relogin=True)
print('W&B ready!')

## CELL 5 — Upload ablation scripts to project
Copy train_ablation.py and wandb_report_visuals.py to the project folder

In [ ]:
# These scripts need to be in your GitHub repo
# If not, upload them as a Kaggle dataset and copy here
import os

PROJECT_DIR = '/kaggle/working/DL6402_Assignment2'

# Check if scripts exist
for script in ['train_ablation.py', 'wandb_report_visuals.py']:
    path = os.path.join(PROJECT_DIR, script)
    if os.path.exists(path):
        print(f'✅ {script} found')
    else:
        print(f'❌ {script} not found — please upload to GitHub repo')

---
## CELL 6 — Sections 2.1 + 2.2: Ablation Runs
⏱ ~2 hours (5 runs × 20 epochs each)

Runs:
- with BN vs without BN (Section 2.1)
- dropout p=0, p=0.2, p=0.5 (Section 2.2)

In [ ]:
import os
os.chdir('/kaggle/working/DL6402_Assignment2')

!python train_ablation.py \
    --data_root /kaggle/working/pets \
    --ckpt_dir  /kaggle/working/checkpoints \
    --epochs    20 \
    --batch_size 64 \
    --lr        3e-4

---
## CELL 7 — Section 2.4: Feature Maps
⏱ ~2 mins

In [ ]:
import os
os.chdir('/kaggle/working/DL6402_Assignment2')

!python wandb_report_visuals.py \
    --data_root /kaggle/working/pets \
    --ckpt_dir  /kaggle/working/checkpoints \
    --img_dir   /kaggle/working/internet_images \
    --sections  2.4

---
## CELL 8 — Section 2.5: BBox Prediction Table
⏱ ~5 mins

In [ ]:
import os
os.chdir('/kaggle/working/DL6402_Assignment2')

!python wandb_report_visuals.py \
    --data_root /kaggle/working/pets \
    --ckpt_dir  /kaggle/working/checkpoints \
    --sections  2.5

---
## CELL 9 — Section 2.6: Segmentation Samples
⏱ ~3 mins

In [ ]:
import os
os.chdir('/kaggle/working/DL6402_Assignment2')

!python wandb_report_visuals.py \
    --data_root /kaggle/working/pets \
    --ckpt_dir  /kaggle/working/checkpoints \
    --sections  2.6

---
## CELL 10 — Upload 3 internet pet images then run Section 2.7

Upload 3 pet images to Kaggle as a dataset named `internet-pets`
with files: pet1.jpg, pet2.jpg, pet3.jpg

Or download them directly in Colab:

In [ ]:
import os

INET_DIR = '/kaggle/working/internet_images'
os.makedirs(INET_DIR, exist_ok=True)

# Download 3 sample pet images from Wikipedia (public domain)
pets = [
    ('pet1.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg'),
    ('pet2.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/b/bb/Kittyply_edit1.jpg/320px-Kittyply_edit1.jpg'),
    ('pet3.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg'),
]

for fname, url in pets:
    fpath = os.path.join(INET_DIR, fname)
    if not os.path.exists(fpath):
        ret = os.system(f'wget -q -O "{fpath}" "{url}"')
        print(f'Downloaded {fname}' if ret==0 else f'Failed {fname}')
    else:
        print(f'{fname} already exists')

print('\nImages in internet_images folder:')
for f in os.listdir(INET_DIR):
    print(' ', f)

In [ ]:
import os
os.chdir('/kaggle/working/DL6402_Assignment2')

!python wandb_report_visuals.py \
    --data_root /kaggle/working/pets \
    --ckpt_dir  /kaggle/working/checkpoints \
    --img_dir   /kaggle/working/internet_images \
    --sections  2.7

---
## CELL 11 — Verify all W&B runs created

In [ ]:
import wandb

api = wandb.Api()
runs = api.runs('da25m019-indian-institute-of-technology-madras/DA6401-A2')

print('All W&B runs in project:')
print(f'{"Run Name":<35} {"State":<12} {"Created"}')
print('-'*65)
for run in runs:
    print(f'{run.name:<35} {run.state:<12} {str(run.created_at)[:10]}')